In [1]:
import pandas as pd

tourism_weather = pd.read_csv(
    "../data/processed/tourism_weather_clean.csv"
)

availability = pd.read_csv(
    "../data/processed/park_availability_clean.csv"
)

print("Tourism + Weather:")
print(tourism_weather.shape)

print("\nAvailability:")
print(availability.shape)

print("\nคอลัมน์ Tourism + Weather:")
print(tourism_weather.columns.tolist())

Tourism + Weather:
(14427, 15)

Availability:
(1980, 12)

คอลัมน์ Tourism + Weather:
['fiscal_year_be', 'park_name_x', 'office', 'month_th', 'visitors', 'month_num', 'calendar_year_be', 'calendar_year_ce', 'date', 'park_key', 'park_key_final', 'park_name_y', 'min_rain', 'max_rain', 'avg_rain']


2. สร้าง Master Dataset

เชื่อมข้อมูลนักท่องเที่ยว ปริมาณฝน และสถานะการเปิด-ปิดของอุทยาน
โดยใช้ชื่ออุทยานและเดือนเป็นตัวเชื่อม
เพื่อสร้าง Dataset หลักสำหรับการวิเคราะห์และ Power BI

In [2]:
master = tourism_weather.merge(
    availability[
        [
            "park_name",
            "month",
            "total_attractions",
            "open_attractions",
            "closed_attractions",
            "unknown_attractions",
            "known_attractions",
            "availability_pct",
            "unknown_pct",
            "availability_data_status"
        ]
    ],
    left_on=["park_name_x", "month_num"],
    right_on=["park_name", "month"],
    how="left"
)

print("ขนาดข้อมูล Master:")
print(master.shape)

print("\nจำนวนข้อมูลซ้ำ:")
print(
    master.duplicated(
        subset=["park_name_x", "date"]
    ).sum()
)

print("\nMissing Availability:")
print(master["availability_data_status"].isna().sum())

master.head()

ขนาดข้อมูล Master:
(14427, 25)

จำนวนข้อมูลซ้ำ:
0

Missing Availability:
0


,fiscal_year_be,park_name_x,office,month_th,visitors,month_num,calendar_year_be,calendar_year_ce,date,park_key,...,park_name,month,total_attractions,open_attractions,closed_attractions,unknown_attractions,known_attractions,availability_pct,unknown_pct,availability_data_status
0,2561,กุยบุรี,สำนักบริหารพื้นที่อนุรักษ์ที่ 3 สาขาเพชรบุรี,ม.ค.,2199,1,2561,2018,2018-01-01,กุยบุรี,...,กุยบุรี,1,2.0,2.0,0.0,0.0,2.0,100.0,0.0,Known
1,2561,ขุนขาน,สำนักบริหารพื้นที่อนุรักษ์ที่ 16,ม.ค.,16242,1,2561,2018,2018-01-01,ขุนขาน,...,ขุนขาน,1,4.0,4.0,0.0,0.0,4.0,100.0,0.0,Known
2,2561,ขุนน่าน,สำนักบริหารพื้นที่อนุรักษ์ที่ 13,ม.ค.,523,1,2561,2018,2018-01-01,ขุนน่าน,...,ขุนน่าน,1,4.0,4.0,0.0,0.0,4.0,100.0,0.0,Known
3,2561,ขุนพะวอ,สำนักบริหารพื้นที่อนุรักษ์ที่ 14,ม.ค.,521,1,2561,2018,2018-01-01,ขุนพะวอ,...,ขุนพะวอ,1,8.0,5.0,0.0,3.0,5.0,100.0,37.5,Known
4,2561,ขุนสถาน,สำนักบริหารพื้นที่อนุรักษ์ที่ 13,ม.ค.,9956,1,2561,2018,2018-01-01,ขุนสถาน,...,ขุนสถาน,1,4.0,4.0,0.0,0.0,4.0,100.0,0.0,Known


3. จัดโครงสร้างและบันทึก Master Dataset

เลือกและจัดชื่อคอลัมน์ที่จำเป็นจากข้อมูลนักท่องเที่ยว
ปริมาณฝน และสถานะการเปิด-ปิด เพื่อสร้าง Dataset หลัก
สำหรับการวิเคราะห์และนำเข้า Power BI

In [3]:
master_clean = master[
    [
        "fiscal_year_be",
        "park_name_x",
        "office",
        "month_th",
        "month_num",
        "calendar_year_be",
        "calendar_year_ce",
        "date",
        "visitors",
        "min_rain",
        "max_rain",
        "avg_rain",
        "total_attractions",
        "open_attractions",
        "closed_attractions",
        "unknown_attractions",
        "known_attractions",
        "availability_pct",
        "unknown_pct",
        "availability_data_status"
    ]
].copy()

master_clean = master_clean.rename(columns={
    "park_name_x": "park_name"
})

master_clean = (
    master_clean
    .sort_values(["date", "park_name"])
    .reset_index(drop=True)
)

master_clean.head()

,fiscal_year_be,park_name,office,month_th,month_num,calendar_year_be,calendar_year_ce,date,visitors,min_rain,max_rain,avg_rain,total_attractions,open_attractions,closed_attractions,unknown_attractions,known_attractions,availability_pct,unknown_pct,availability_data_status
0,2561,กุยบุรี,สำนักบริหารพื้นที่อนุรักษ์ที่ 3 สาขาเพชรบุรี,ม.ค.,1,2561,2018,2018-01-01,2199,14.5,54.709999,28.620681,2.0,2.0,0.0,0.0,2.0,100.0,0.0,Known
1,2561,ขุนขาน,สำนักบริหารพื้นที่อนุรักษ์ที่ 16,ม.ค.,1,2561,2018,2018-01-01,16242,0.3,30.400000,6.695410,4.0,4.0,0.0,0.0,4.0,100.0,0.0,Known
2,2561,ขุนน่าน,สำนักบริหารพื้นที่อนุรักษ์ที่ 13,ม.ค.,1,2561,2018,2018-01-01,523,11.4,30.780001,17.231855,4.0,4.0,0.0,0.0,4.0,100.0,0.0,Known
3,2561,ขุนพะวอ,สำนักบริหารพื้นที่อนุรักษ์ที่ 14,ม.ค.,1,2561,2018,2018-01-01,521,0.0,32.000000,9.917311,8.0,5.0,0.0,3.0,5.0,100.0,37.5,Known
4,2561,ขุนสถาน,สำนักบริหารพื้นที่อนุรักษ์ที่ 13,ม.ค.,1,2561,2018,2018-01-01,9956,11.4,30.780001,17.231855,4.0,4.0,0.0,0.0,4.0,100.0,0.0,Known


In [4]:
print("ขนาด Master Clean:")
print(master_clean.shape)

print("\nคอลัมน์:")
print(master_clean.columns.tolist())

print("\nMissing Values:")
print(master_clean.isna().sum())

ขนาด Master Clean:
(14427, 20)

คอลัมน์:
['fiscal_year_be', 'park_name', 'office', 'month_th', 'month_num', 'calendar_year_be', 'calendar_year_ce', 'date', 'visitors', 'min_rain', 'max_rain', 'avg_rain', 'total_attractions', 'open_attractions', 'closed_attractions', 'unknown_attractions', 'known_attractions', 'availability_pct', 'unknown_pct', 'availability_data_status']

Missing Values:
fiscal_year_be               0
park_name                    0
office                       0
month_th                     0
month_num                    0
calendar_year_be             0
calendar_year_ce             0
date                         0
visitors                     0
min_rain                     0
max_rain                     0
avg_rain                     0
total_attractions           12
open_attractions            12
closed_attractions          12
unknown_attractions         12
known_attractions           12
availability_pct            12
unknown_pct                 12
availability_data_st

In [5]:
master_clean.to_csv(
    "../data/processed/national_park_master.csv",
    index=False,
    encoding="utf-8-sig"
)

print("บันทึก national_park_master.csv เรียบร้อย")
print(master_clean.shape)

บันทึก national_park_master.csv เรียบร้อย
(14427, 20)


4. สร้าง Park Dimension

สร้างตารางข้อมูลอุทยานแบบหนึ่งแถวต่อหนึ่งอุทยาน
สำหรับใช้เป็น Dimension Table ในฐานข้อมูลและ Power BI

ประกอบด้วยชื่ออุทยาน สำนักงาน จังหวัด และพิกัดตัวแทนของอุทยาน
โดยพิกัดคำนวณจากค่ากลางของแหล่งท่องเที่ยวภายในอุทยาน

In [6]:
import re
import numpy as np

# โหลดข้อมูลเปิด-ปิดที่มีจังหวัดและพิกัด
try:
    status_raw = pd.read_csv(
        "../data/openclosenppark.csv",
        encoding="utf-8-sig"
    )
except UnicodeDecodeError:
    status_raw = pd.read_csv(
        "../data/openclosenppark.csv",
        encoding="cp874"
    )

status_raw.columns = status_raw.columns.str.strip()


# Function ปรับชื่ออุทยานให้เหมือนที่ใช้ก่อนหน้า
def normalize_park_name(name):
    if pd.isna(name):
        return None

    name = str(name).strip()
    name = pd.Series([name]).str.normalize("NFC").iloc[0]

    name = re.sub(r"^อุทยานแห่งชาติ\s*", "", name)
    name = re.sub(r"\s*\(เตรียมการฯ?\)", "", name)
    name = name.replace("\u200b", "")
    name = re.sub(r"\s*[-–—]\s*", "-", name)
    name = re.sub(r"\s+", " ", name)

    return name.strip()


status_raw["park_status_key"] = (
    status_raw["ชื่อหน่วยงาน"]
    .apply(normalize_park_name)
)

# พิกัดบางช่องเป็น string → แปลงเป็นตัวเลข
status_raw["latitude"] = pd.to_numeric(
    status_raw["latitude"],
    errors="coerce"
)

status_raw["longitude"] = pd.to_numeric(
    status_raw["longitude"],
    errors="coerce"
)

In [7]:
# ใช้จังหวัดที่พบมากที่สุดของแต่ละอุทยาน
def get_mode(series):
    values = series.dropna().astype(str).str.strip()

    if len(values) == 0:
        return np.nan

    mode = values.mode()

    if len(mode) > 0:
        return mode.iloc[0]

    return values.iloc[0]


# สรุป Location ระดับอุทยาน
park_location = (
    status_raw
    .groupby("park_status_key")
    .agg(
        province=("จังหวัด", get_mode),
        latitude=("latitude", "median"),
        longitude=("longitude", "median")
    )
    .reset_index()
)


# ตารางเชื่อมชื่อ Tourism กับชื่อในข้อมูลเปิด-ปิด
park_bridge = (
    availability[
        ["park_name", "park_status_final"]
    ]
    .drop_duplicates()
)


# เอาสำนักงานล่าสุดของแต่ละอุทยาน
current_office = (
    master_clean
    .sort_values("date")
    .groupby("park_name")
    .tail(1)[
        ["park_name", "office"]
    ]
)


# รวมเป็น Dimension Table
dim_park = (
    park_bridge
    .merge(
        park_location,
        left_on="park_status_final",
        right_on="park_status_key",
        how="left"
    )
    .merge(
        current_office,
        on="park_name",
        how="left"
    )
)


# แม่ยวมฝั่งซ้ายไม่มีข้อมูลในชุดเปิด-ปิด
# แต่เรารู้จังหวัดจาก mapping ก่อนหน้า
dim_park.loc[
    dim_park["park_name"] == "แม่ยวมฝั่งซ้าย",
    "province"
] = "แม่ฮ่องสอน"


# เลือกเฉพาะคอลัมน์ที่ใช้จริง
dim_park = dim_park[
    [
        "park_name",
        "province",
        "office",
        "latitude",
        "longitude"
    ]
].sort_values("park_name").reset_index(drop=True)

In [8]:
print("ขนาด dim_park:")
print(dim_park.shape)

print("\nจำนวนอุทยาน:")
print(dim_park["park_name"].nunique())

print("\nจังหวัดที่ Missing:")
print(dim_park["province"].isna().sum())

print("\nLatitude ที่ Missing:")
print(dim_park["latitude"].isna().sum())

print("\nLongitude ที่ Missing:")
print(dim_park["longitude"].isna().sum())

display(
    dim_park[
        dim_park["latitude"].isna() |
        dim_park["longitude"].isna()
    ]
)

ขนาด dim_park:
(165, 5)

จำนวนอุทยาน:
165

จังหวัดที่ Missing:
1

Latitude ที่ Missing:
3

Longitude ที่ Missing:
3


,park_name,province,office,latitude,longitude
81,ภูหินร่องกล้า,พิษณุโลก,สำนักบริหารพื้นที่อนุรักษ์ที่ 11 (พิษณุโลก),NaN,NaN
152,แม่ยวมฝั่งซ้าย,แม่ฮ่องสอน,สำนักบริหารพื้นที่อนุรักษ์ที่ 16 สาขาแม่สะเรียง,NaN,NaN
160,แหลมสน,NaN,สำนักบริหารพื้นที่อนุรักษ์ที่ 4 (สุราษฎร์ธานี),NaN,NaN


In [9]:
dim_park.loc[
    dim_park["park_name"] == "แหลมสน",
    "province"
] = "ระนอง"

In [10]:
dim_park.to_csv(
    "../data/processed/dim_park.csv",
    index=False,
    encoding="utf-8-sig"
)

print("บันทึก dim_park.csv เรียบร้อย")
print(dim_park.shape)

บันทึก dim_park.csv เรียบร้อย
(165, 5)
